**Project Name**

***Identifying and Quantifying Piezoelectric Energy Potential on Austrian Railways***

**Project goal**

Find the most suitable railway stretches in Austria for installing piezoelectric harvesters, based on train frequency, type, and estimated mechanical load. Then estimate how much energy could be harvested yearly.

**Analytical workflow**

**Train frequency analysis**

Parse GTFS → count trains per station pair (e.g., Wien–St. Pölten).

Rank top 10 busiest segments.

**Mechanical load estimation**

Assign average axle load per train type (e.g., Railjet ≈ 17 t, S-Bahn ≈ 12 t).

Compute average total load per day for each segment.

**Energy model**

Use simple model or empirical formulas from published road/rail piezo studies (e.g., 5–20 mJ per axle event per unit).

**Aggregate by segment length → total daily/annual kWh per km of track.**

**Visualize → heatmap of Austria’s rail network showing estimated energy potential.**

**(Optional) Compare scenarios: passenger vs freight, weekdays vs weekends.**

**Fetching the data**

In [1]:
# ============================================================
# 1. Set up file paths and dataset URL
# ============================================================

from pathlib import Path
import os
import requests
from zipfile import ZipFile
import pandas as pd
import random


# Folder where we'll store the GTFS data
DATA_DIR = Path("data/oebb_gtfs")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# URL of the ÖBB GTFS dataset (update if new version is released)
GTFS_URL = "https://static.web.oebb.at/open-data/soll-fahrplan-gtfs/GTFS_OP_2025_obb.zip"

# Local ZIP file and extraction folder
ZIP_PATH = DATA_DIR / "oebb_gtfs.zip"
EXTRACTED_FLAG = DATA_DIR / "routes.txt"  # we'll check for one key file to see if it's extracted

print("Data directory:", DATA_DIR)

# ============================================================
# 2. Check if dataset already exists before downloading
# ============================================================

if ZIP_PATH.exists() and EXTRACTED_FLAG.exists():
    print("\n GTFS data already downloaded and extracted.")
    print("   Skipping download...")
else:
    print("\n Downloading ÖBB GTFS dataset...")
    response = requests.get(GTFS_URL)
    response.raise_for_status()  # raise error if something fails

    # Save ZIP file locally
    with open(ZIP_PATH, "wb") as f:
        f.write(response.content)
    print(" Download complete! Saved to:", ZIP_PATH)

# ============================================================
# 3. Extract the ZIP archive
# ============================================================
    print("\n Extracting GTFS files...")
    with ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    print(" Files extracted to:", DATA_DIR)

print("\n Setup complete — ready to load the GTFS tables.")
# ============================================================

Data directory: data/oebb_gtfs

 GTFS data already downloaded and extracted.
   Skipping download...

 Setup complete — ready to load the GTFS tables.


**Parse the data**

In [2]:
# ============================================================
# 4. Load main GTFS tables into pandas DataFrames
# ============================================================

# The main files we'll use
FILES = [
    "routes.txt",
    "trips.txt",
    "stop_times.txt",
    "stops.txt",
    "calendar.txt"
]

# Load each file if it exists
dfs = {}
for file in FILES:
    file_path = DATA_DIR / file
    if file_path.exists():
        dfs[file.split('.')[0]] = pd.read_csv(file_path)
        print(f"Loaded {file}: {dfs[file.split('.')[0]].shape[0]} rows")
    else:
        print(f"Missing {file}")

# ============================================================
# 5. Quick preview of key tables
# ============================================================

print("\n routes.txt sample:")
print(dfs['routes'].head(), "\n")

print(" trips.txt sample:")
print(dfs['trips'].head(), "\n")

print(" calendar.txt sample:")
print(dfs['calendar'].head(), "\n")

# ============================================================
# Dataset successfully loaded and ready for analysis!
# ============================================================
print("ÖBB GTFS dataset is ready for analysis!")

Loaded routes.txt: 246 rows
Loaded trips.txt: 9615 rows
Loaded stop_times.txt: 109757 rows
Loaded stops.txt: 7141 rows
Loaded calendar.txt: 714 rows

 routes.txt sample:
       route_id  agency_id route_short_name route_long_name  route_type
0  10-A10-j25-1          1              A10             NaN           2
1  10-A11-j25-1          1              A11             NaN           2
2  10-A12-j25-1          1              A12             NaN           2
3  10-A13-j25-1          1              A13             NaN           2
4  10-A14-j25-1          1              A14             NaN           2 

 trips.txt sample:
       route_id service_id                 trip_id          shape_id  \
0  10-A10-j25-1   TA+gvw20   1.TA.10-A10-j25-1.1.R  10-A10-j25-1.1.R   
1  10-A10-j25-1   TA+evw20   2.TA.10-A10-j25-1.2.R  10-A10-j25-1.2.R   
2  10-A11-j25-1   TA+07y20   1.TA.10-A11-j25-1.1.R  10-A11-j25-1.1.R   
3  10-A11-j25-1   TA+nqw20  10.TA.10-A11-j25-1.3.H  10-A11-j25-1.3.H   
4  10-A11-j25-1  

**Goal:**

Connect each trip (individual scheduled train) with its corresponding route information (train line name, operator, type).

**Why:**

This lets us know which line each scheduled train belongs to — for example, Railjet, S-Bahn, REX, etc.

**trips.txt** contains one row per train run.

Key columns: trip_id, route_id, service_id, trip_headsign

**routes.txt** contains metadata about each line.

Key columns: route_id, route_short_name, route_long_name, route_type

We’ll join them on route_id.

In [3]:
# ============================================================
# Merge TRIPS with ROUTES
# ============================================================

trips_routes = dfs['trips'].merge(
    dfs['routes'],
    on='route_id',
    how='left'
)

print(f" Merged dataset shape: {trips_routes.shape}")
print("\n Columns available:", trips_routes.columns.tolist())

# Show a few sample rows
trips_routes.head(5)


 Merged dataset shape: (9615, 12)

 Columns available: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']


,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id,agency_id,route_short_name,route_long_name,route_type
0,10-A10-j25-1,TA+gvw20,1.TA.10-A10-j25-1.1.R,10-A10-j25-1.1.R,Villach Hauptbahnhof,RJ 639,1,NaN,1,A10,NaN,2
1,10-A10-j25-1,TA+evw20,2.TA.10-A10-j25-1.2.R,10-A10-j25-1.2.R,Spittal-Millstätter See Bahnhof,RJ 639,1,NaN,1,A10,NaN,2
2,10-A11-j25-1,TA+07y20,1.TA.10-A11-j25-1.1.R,10-A11-j25-1.1.R,Budapest-Keleti,RJX 165,1,430.0,1,A11,NaN,2
3,10-A11-j25-1,TA+nqw20,10.TA.10-A11-j25-1.3.H,10-A11-j25-1.3.H,Zürich HB,RJX 264,0,NaN,1,A11,NaN,2
4,10-A11-j25-1,TA+fww20,11.TA.10-A11-j25-1.4.H,10-A11-j25-1.4.H,Salzburg Hauptbahnhof,RJX 264,0,NaN,1,A11,NaN,2


Link calendar to determine service days

**Why this matters**

Every trip (train run) has a **service_id**.

The calendar.txt file tells us what days of the week that service runs.

So, by merging calendar onto trips_routes, we can later count how many trains operate per route on each day of the week.

In [4]:
# ============================================================
# Link CALENDAR to determine service days
# ============================================================

# Merge with calendar on service_id
trips_routes_cal = trips_routes.merge(
    dfs['calendar'],
    on='service_id',
    how='left'
)

print(f"Merged dataset shape: {trips_routes_cal.shape}")
print("\nColumns now available:", trips_routes_cal.columns.tolist())

# Show sample rows with day-of-week columns
cols_to_show = [
    'route_id', 'route_short_name', 'route_long_name', 
    'trip_id', 'service_id', 'monday', 'tuesday', 'wednesday', 
    'thursday', 'friday', 'saturday', 'sunday'
]

trips_routes_cal[cols_to_show].head(5)


Merged dataset shape: (9615, 21)

Columns now available: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,route_id,route_short_name,route_long_name,trip_id,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday
0,10-A10-j25-1,A10,NaN,1.TA.10-A10-j25-1.1.R,TA+gvw20,1,1,1,1,1,1,1
1,10-A10-j25-1,A10,NaN,2.TA.10-A10-j25-1.2.R,TA+evw20,1,1,1,1,1,1,1
2,10-A11-j25-1,A11,NaN,1.TA.10-A11-j25-1.1.R,TA+07y20,0,0,0,0,0,1,1
3,10-A11-j25-1,A11,NaN,10.TA.10-A11-j25-1.3.H,TA+nqw20,0,0,0,0,0,1,0
4,10-A11-j25-1,A11,NaN,11.TA.10-A11-j25-1.4.H,TA+fww20,1,1,1,1,1,0,1


In [5]:
# ============================================================
# Count trips per day per route
# ============================================================

# Select relevant columns for aggregation
cols = [
    'route_id', 'route_short_name', 'route_long_name',
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'
]

# Group by route (each row = 1 trip), sum the 0/1 flags to count trips per day
route_freq = (
    trips_routes_cal[cols]
    .groupby(['route_id', 'route_short_name', 'route_long_name'], as_index=False)
    .sum(numeric_only=True)
)

# Optional: compute total weekday and total weekly trips
route_freq['trips_weekday_total'] = (
    route_freq[['monday','tuesday','wednesday','thursday','friday']].sum(axis=1)
)
route_freq['trips_week_total'] = (
    route_freq[['monday','tuesday','wednesday','thursday','friday','saturday','sunday']].sum(axis=1)
)

# Sort by total weekly trips (descending)
route_freq = route_freq.sort_values('trips_week_total', ascending=False)

print(f" Calculated frequencies for {len(route_freq)} routes.")
route_freq.head(10)


 Calculated frequencies for 139 routes.


,route_id,route_short_name,route_long_name,monday,tuesday,wednesday,thursday,friday,saturday,sunday,trips_weekday_total,trips_week_total
23,1-S45-W-j25-1,S45,S45,227,227,227,227,227,229,226,1135,1590
6,1-S1-W-j25-1,S1,S1,169,169,170,169,166,184,179,843,1206
26,1-S50-W-j25-1,S50,S50,141,141,141,141,141,136,138,705,979
11,1-S2-W-j25-1,S2,S2,126,126,126,126,126,125,121,630,876
84,2-RX8-W-j25-1,REX8,REX8,116,116,116,116,116,116,116,580,812
17,1-S3-W-j25-1,S3,S3,126,129,129,129,127,82,82,640,804
107,3-R44-N-j25-1,R44,R44,112,112,112,112,112,86,86,560,732
2,1-S1-K-j25-1,S1,S1,113,112,112,112,112,70,62,561,693
22,1-S40-W-j25-1,S40,S40,99,99,99,99,99,99,97,495,691
69,2-RX1-W-j25-1,REX1,REX1,100,100,100,100,100,84,81,500,665


For every trip (trip_id):

Find the first stop (stop_sequence = smallest)

Find the last stop (stop_sequence = largest)

Add the readable stop names from stops.txt

We’ll then merge these with the route/trip/calendar table (trips_routes_cal)
→ giving you direction + line + active days.

In [6]:
# ============================================================
# Extract START and END stations per trip
# ============================================================

# Merge stop_times with stops to get readable station names
stop_times_named = dfs['stop_times'].merge(
    dfs['stops'][['stop_id', 'stop_name']],
    on='stop_id',
    how='left'
)

# Sort by trip and stop sequence
stop_times_named = stop_times_named.sort_values(['trip_id', 'stop_sequence'])

# Extract first and last stop per trip
first_stops = stop_times_named.groupby('trip_id').first().reset_index()
last_stops  = stop_times_named.groupby('trip_id').last().reset_index()

# Create a simplified DataFrame with trip_id, start_station, end_station
trip_endpoints = pd.DataFrame({
    'trip_id': first_stops['trip_id'],
    'start_station': first_stops['stop_name'],
    'end_station': last_stops['stop_name']
})

print(f"Extracted start and end stations for {len(trip_endpoints)} trips.")
trip_endpoints.head()


Extracted start and end stations for 9615 trips.


,trip_id,start_station,end_station
0,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof
1,1.TA.1-MS1-V-j25-1.1.R,Bludenz Bahnhof,Bregenz Bahnhof
2,1.TA.1-S1-K-j25-1.1.R,Villach Hauptbahnhof,Föderlach Bahnhof
3,1.TA.1-S1-M-j25-1.1.R,Graz Hbf 4,Frohnleiten Bahnhof 1
4,1.TA.1-S1-O-j25-1.1.R,St. Valentin Bahnhof,Linz/Donau Hauptbahnhof


Now we can merge this trip_endpoints table with the trips_routes_cal dataset we built earlier to include route info + direction + service days.

In [7]:
# ============================================================
# Merge endpoints with route/calendar info
# ============================================================

trips_full = trips_routes_cal.merge(
    trip_endpoints,
    on='trip_id',
    how='left'
)

print(f" Merged full trip dataset shape: {trips_full.shape}")
trips_full[['route_short_name', 'trip_id', 'start_station', 'end_station']].head()


 Merged full trip dataset shape: (9615, 23)


,route_short_name,trip_id,start_station,end_station
0,A10,1.TA.10-A10-j25-1.1.R,Graz Hbf 2,Villach Hauptbahnhof
1,A10,2.TA.10-A10-j25-1.2.R,Graz Hbf 2,Spittal-Millstätter See Bahnhof
2,A11,1.TA.10-A11-j25-1.1.R,Kufstein Bahnhof,Hegyeshalom pályaudvar
3,A11,10.TA.10-A11-j25-1.3.H,Budapest-Keleti,Wien Hauptbahnhof
4,A11,11.TA.10-A11-j25-1.4.H,Budapest-Keleti,Salzburg Hauptbahnhof


Count number of trips per Origin–Destination (OD) pair

**Goal**

Group all trips by:

start_station

end_station

Then count how many trains (trips) operate on each OD pair.

Optionally, we can include day-of-week info (to see if it changes).

In [2]:
# ============================================================
# Count number of trips per origin–destination pair
# ============================================================
import pandas as pd

trips_full = pd.read_csv("data/oebb_gtfs/stop_times_geo_with_startEnd_calc.csv")


# Select relevant columns
cols = [
    'start_station', 'end_station',
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday',
    'saturday', 'sunday'
]

# Group by OD pair, sum the 0/1 flags for trips operating that day
od_counts = (
    trips_full[cols]
    .groupby(['start_station', 'end_station'], as_index=False)
    .sum(numeric_only=True)
)

# Optional: compute total weekday & total weekly trips
od_counts['weekday_total'] = od_counts[['monday','tuesday','wednesday','thursday','friday']].sum(axis=1)
od_counts['weekly_total'] = od_counts[['monday','tuesday','wednesday','thursday','friday','saturday','sunday']].sum(axis=1)

# Sort by total weekly trips
#od_counts = od_counts.sort_values('weekly_total', ascending=False)

output_path = "data/oebb_gtfs/stop_times_geo_with_startEnd_calc1.csv"
trips_full.to_csv(output_path, index=False)


print(f" Computed OD trip counts for {len(od_counts)} start–end pairs.")
od_counts.head(10)


 Computed OD trip counts for 1080 start–end pairs.


,start_station,end_station,monday,tuesday,wednesday,thursday,friday,saturday,sunday,weekday_total,weekly_total
0,Absdorf-Hippersdorf Bahnhof,Leobersdorf Bahnhof,175,175,175,175,175,35,35,875,945
1,Absdorf-Hippersdorf Bahnhof,Mödling Bahnhof,112,112,112,112,112,280,308,560,1148
2,Absdorf-Hippersdorf Bahnhof,Stockerau Bahnhof,4,4,4,4,4,4,4,20,28
3,Absdorf-Hippersdorf Bahnhof,Wiener Neustadt Hauptbahnhof,154,154,154,154,154,38,0,770,808
4,Admont Bahnhof,Attnang-Puchheim Bahnhof,0,0,0,0,0,36,36,0,72
5,Admont Bahnhof,Selzthal Bahnhof,0,0,0,0,0,2,2,0,4
6,Aigen-Schlägl Bahnhof,Linz/Donau Urfahr Bahnhof,179,179,179,179,179,179,149,895,1223
7,Amstetten Bahnhof,Kastenreith Bahnhst,84,84,84,84,84,0,0,420,420
8,Amstetten Bahnhof,Kleinreifling Bahnhof,105,105,105,105,90,75,90,510,675
9,Amstetten Bahnhof,Linz/Donau Hauptbahnhof,3,3,3,3,3,0,0,15,15


In [9]:
# Inspect distinct route_short_name values
trips_full['route_short_name'].dropna().unique()[:40]


array(['A10', 'A11', 'A12', 'A13', 'A14', 'A18', 'A19', 'A1', 'A21',
       'A23', 'A3', 'A4', 'A62', 'A6', 'A7', 'A8', 'A92', 'A9', 'CH3',
       'D17', 'D9', 'HU1', 'SI1', 'SK', '220-A', 'A15', 'A-1', 'A85',
       'A86', 'A88', 'A91', 'A94', 'A99', 'CH1', 'CH2', 'CZ1', 'CZ3',
       'CZ4', 'D10', 'D11'], dtype=object)

In [10]:
dfs['routes'].head(10)


,route_id,agency_id,route_short_name,route_long_name,route_type
0,10-A10-j25-1,1,A10,NaN,2
1,10-A11-j25-1,1,A11,NaN,2
2,10-A12-j25-1,1,A12,NaN,2
3,10-A13-j25-1,1,A13,NaN,2
4,10-A14-j25-1,1,A14,NaN,2
5,10-A18-j25-1,1,A18,NaN,2
6,10-A19-j25-1,1,A19,NaN,2
7,10-A1-j25-1,1,A1,NaN,2
8,10-A21-j25-1,1,A21,NaN,2
9,10-A23-j25-1,1,A23,NaN,2


In [11]:
df_routes=dfs['routes']
df_routes.head()

,route_id,agency_id,route_short_name,route_long_name,route_type
0,10-A10-j25-1,1,A10,NaN,2
1,10-A11-j25-1,1,A11,NaN,2
2,10-A12-j25-1,1,A12,NaN,2
3,10-A13-j25-1,1,A13,NaN,2
4,10-A14-j25-1,1,A14,NaN,2


**Adding weight per train type**

In [1]:
# ============================================================
# Assign train weights based on route_short_name
# ============================================================
import pandas as pd

def estimate_weight_from_short(route):
    """Estimate train mass (tonnes) based on route_short_name pattern."""
    if pd.isna(route):
        return 220  # fallback
    
    name = str(route).upper().strip()

    # --- Commuter / S-Bahn type ---
    if name.startswith("S") or name.startswith("CAT"):   # CAT = City Airport Train
        return 160

    # --- Regional services (REX, RX, R, SV etc.) ---
    if any(name.startswith(prefix) for prefix in ["REX", "RX", "R", "SV"]):
        return 250

    # --- Intercity / Long-distance (CJX, RJ, IC, EC, EN, WB, A*, D*, CH*, CZ*, HU*, SI*) ---
    if any(name.startswith(prefix) for prefix in [
        "RJ", "RJX", "CJX", "IC", "EC", "EN", "WB", "A", "D", "CH", "CZ", "HU", "SI"
    ]):
        return 400

    # --- Bus-like or special services (ICBus, etc.) ---
    if "BUS" in name:
        return 150

    # --- Default ---
    return 220


# Apply mapping
trips_full = pd.read_csv("data/oebb_gtfs/stop_times_geo_with_startEnd_calc.csv")
trips_full['estimated_weight_t'] = trips_full['route_short_name'].apply(estimate_weight_from_short)

# Inspect unique mapping result
print("Added estimated train weight (tonnes) based on route_short_name pattern.")
print(trips_full[['route_short_name', 'estimated_weight_t']].drop_duplicates().sort_values('route_short_name').head(40))

# Quick distribution check
print("\nWeight distribution:")
print(trips_full['estimated_weight_t'].value_counts())

output_path = "data/oebb_gtfs/stop_times_geo_with_startEnd_calc.csv"
trips_full.to_csv(output_path, index=False)

print(f"Full dataset exported to: {output_path}")
print(f"Contains {len(trips_full):,} rows and {len(trips_full.columns)} columns.")


Added estimated train weight (tonnes) based on route_short_name pattern.
      route_short_name  estimated_weight_t
588              101-D                 220
1249             103-D                 220
1253             130-O                 220
1259             140-O                 220
1268             150-O                 220
1270             190-S                 220
440              220-A                 220
769              220-K                 220
771              221-S                 220
1289             251-S                 220
776              302-A                 220
1293             499-M                 220
780              501-M                 220
785              502-M                 220
1299             520-N                 220
43917            524-A                 220
793              530-M                 220
806              600-M                 220
1309             620-K                 220
1311             810-W                 220
818              902-W  

Adding distance and average speed introduces kinematic realism: how much work is being done over that segment, which directly affects how much mechanical stress (and thus piezoelectric energy) is generated.

In [13]:
# ============================================================
# Compute distance and average speed per trip
# ============================================================

import numpy as np

# --- Helper function to compute haversine distance ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Merge stop_times with stops for coordinates
stop_times_geo = dfs['stop_times'].merge(
    dfs['stops'][['stop_id', 'stop_lat', 'stop_lon']],
    on='stop_id', how='left'
)

# Extract first and last stops for each trip
first_stops = stop_times_geo.sort_values(['trip_id', 'stop_sequence']).groupby('trip_id').first().reset_index()
last_stops  = stop_times_geo.sort_values(['trip_id', 'stop_sequence']).groupby('trip_id').last().reset_index()

# Compute distance (km)
distances = haversine(first_stops['stop_lat'], first_stops['stop_lon'],
                      last_stops['stop_lat'], last_stops['stop_lon'])

# --- Compute travel time ---
# Convert HH:MM:SS to seconds (handles times > 24:00)
def time_to_seconds(t):
    h, m, s = map(int, t.split(':'))
    return h*3600 + m*60 + s

start_time = first_stops['departure_time'].apply(time_to_seconds)
end_time = last_stops['arrival_time'].apply(time_to_seconds)
duration_sec = end_time - start_time

# Handle cases where arrival time rolls past midnight
duration_sec[duration_sec < 0] += 24*3600

# Compute average speed (km/h)
duration_hr = duration_sec / 3600
avg_speed = distances / duration_hr.replace(0, np.nan)  # avoid div/0

# Build table
trip_distance_speed = pd.DataFrame({
    'trip_id': first_stops['trip_id'],
    'distance_km': distances,
    'duration_hr': duration_hr,
    'avg_speed_kmh': avg_speed
})

print(f"Calculated distance and speed for {len(trip_distance_speed)} trips.")
trip_distance_speed.head()


Calculated distance and speed for 9615 trips.


,trip_id,distance_km,duration_hr,avg_speed_kmh
0,1.TA.1-MB4-j25-1.1.R,11.443951,0.316667,36.138792
1,1.TA.1-MS1-V-j25-1.1.R,39.130347,1.083333,36.120320
2,1.TA.1-S1-K-j25-1.1.R,8.627527,0.133333,64.706453
3,1.TA.1-S1-M-j25-1.1.R,22.584050,0.416667,54.201720
4,1.TA.1-S1-O-j25-1.1.R,21.054200,0.383333,54.923999


Merging with the full dataset

In [14]:
# ============================================================
# Merge distance/speed info into trip dataset
# ============================================================

trips_full = trips_full.merge(
    trip_distance_speed,
    on='trip_id',
    how='left'
)

print("Added distance and average speed to trip dataset.")
trips_full[['route_short_name', 'start_station', 'end_station', 'distance_km', 'avg_speed_kmh']].head()

output_path = "data/oebb_gtfs/trips_full_enriched.csv"
trips_full.to_csv(output_path, index=False)

print(f"Full dataset exported to: {output_path}")
print(f"Contains {len(trips_full):,} rows and {len(trips_full.columns)} columns.")



Added distance and average speed to trip dataset.
Full dataset exported to: data/oebb_gtfs/trips_full_enriched.csv
Contains 9,615 rows and 27 columns.


In [15]:
# ============================================================
# Aggregated OD summary table
# ============================================================

# Example: average speed, weight, and number of trips per OD pair
od_summary = (
    trips_full.groupby(['start_station', 'end_station'], as_index=False)
    .agg({
        'estimated_weight_t': 'mean',
        'distance_km': 'mean',
        'avg_speed_kmh': 'mean'
    })
)

output_summary_path = "data/oebb_gtfs/od_summary.csv"
od_summary.to_csv(output_summary_path, index=False)

print(f"OD summary exported to: {output_summary_path}")
print(f"Contains {len(od_summary):,} origin–destination pairs.")


OD summary exported to: data/oebb_gtfs/od_summary.csv
Contains 1,080 origin–destination pairs.


In [16]:
# ============================================================
# Merge trip info (weight/speed) with frequency data
# ============================================================

# Merge 'weekly_total' (frequency) into trips_full using start–end stations
trips_with_freq = trips_full.merge(
    od_counts[['start_station', 'end_station', 'weekly_total']],
    on=['start_station', 'end_station'],
    how='left'
)

# Now group by OD pair to get average weight, speed, distance, and total frequency
top_candidates = (
    trips_with_freq.groupby(['start_station', 'end_station'], as_index=False)
    .agg({
        'estimated_weight_t': 'mean',
        'distance_km': 'mean',
        'avg_speed_kmh': 'mean',
        'weekly_total': 'mean'  # same for all trips within OD, so mean = total
    })
    .sort_values('weekly_total', ascending=False)
)

# Export to CSV
output_path = "data/oebb_gtfs/top_candidates.csv"
top_candidates.to_csv(output_path, index=False)

print(f"Top 50 OD pairs with frequencies exported to: {output_path}")
print(top_candidates.head(10))


Top 50 OD pairs with frequencies exported to: data/oebb_gtfs/top_candidates.csv
               start_station             end_station  estimated_weight_t  \
874          Wien Handelskai         Wien Hütteldorf          160.000000   
921          Wien Hütteldorf         Wien Handelskai          160.000000   
902        Wien Hauptbahnhof     Passau Hauptbahnhof          400.000000   
536      Passau Hauptbahnhof       Wien Hauptbahnhof          400.000000   
321   Innsbruck Hauptbahnhof        Kufstein Bahnhof          349.696970   
382         Kufstein Bahnhof  Innsbruck Hauptbahnhof          350.645161   
837         Wien Aspern Nord         Wien Hütteldorf          160.000000   
920          Wien Hütteldorf        Wien Aspern Nord          160.000000   
466         Marchegg Bahnhof       Wien Hauptbahnhof          250.000000   
688  St. Pölten Hauptbahnhof        Wien Westbahnhof          256.976744   

     distance_km  avg_speed_kmh  weekly_total  
874    10.412476      22.756270    

**Energy Potential Index (EPI)**

A synthetic indicator of how much piezoelectric energy could be harvested per route.

In [17]:
epi_factor = 1e-6  # scale factor to keep numbers manageable
top_candidates['EPI'] = (
    epi_factor
    * top_candidates['weekly_total']
    * top_candidates['estimated_weight_t']
    * (top_candidates['avg_speed_kmh'] ** 2)
    / top_candidates['distance_km']
)

top_candidates = top_candidates.sort_values('EPI', ascending=False)
top_candidates.head(10)

# Export to CSV
output_path = "data/oebb_gtfs/top_candidates.csv"
top_candidates.to_csv(output_path, index=False)



**Adding coordinates for geospatial representation**

In [18]:
# ============================================================
# Add coordinates for start and end stations
# ============================================================

# Load stops table if not already in memory
stops = dfs['stops'][['stop_name', 'stop_lat', 'stop_lon']].drop_duplicates()

# Merge start station coordinates
top_candidates = top_candidates.merge(
    stops.rename(columns={'stop_name': 'start_station',
                          'stop_lat': 'start_lat',
                          'stop_lon': 'start_lon'}),
    on='start_station',
    how='left'
)

# Merge end station coordinates
top_candidates = top_candidates.merge(
    stops.rename(columns={'stop_name': 'end_station',
                          'stop_lat': 'end_lat',
                          'stop_lon': 'end_lon'}),
    on='end_station',
    how='left'
)

print("Coordinates added to OD pairs.")
print(top_candidates[['start_station', 'start_lat', 'start_lon', 'end_station', 'end_lat', 'end_lon']].head())

# Export to CSV
output_path = "data/oebb_gtfs/top_candidates.csv"
top_candidates.to_csv(output_path, index=False)



Coordinates added to OD pairs.
                        start_station  start_lat  start_lon  \
0  Spielfeld Tarifpunkt Grenze (Bahn)   46.69202  15.640540   
1  Spielfeld Tarifpunkt Grenze (Bahn)   46.69202  15.640540   
2  Spielfeld Tarifpunkt Grenze (Bahn)   46.69202  15.640540   
3  Spielfeld Tarifpunkt Grenze (Bahn)   46.69202  15.640540   
4  Spielfeld Tarifpunkt Grenze (Bahn)   46.69202  15.640361   

               end_station    end_lat    end_lon  
0  Spielfeld-Straß Bahnhof  46.709652  15.628853  
1  Spielfeld-Straß Bahnhof  46.709584  15.629042  
2  Spielfeld-Straß Bahnhof  46.709627  15.629159  
3  Spielfeld-Straß Bahnhof  46.709695  15.629204  
4  Spielfeld-Straß Bahnhof  46.709652  15.628853  


**Visual Representation of the routes**

In [19]:
import folium

m = folium.Map(location=[47.5, 14.5], zoom_start=7)  # Center on Austria

for _, row in top_candidates.dropna(subset=['start_lat', 'end_lat']).iterrows():
    folium.PolyLine(
        locations=[(row.start_lat, row.start_lon), (row.end_lat, row.end_lon)],
        color='red',
        weight=row['EPI'] / top_candidates['EPI'].max() * 5 + 1,
        tooltip=f"{row.start_station} → {row.end_station}\nEPI={row.EPI:.2f}"
    ).add_to(m)

m.save("data/oebb_gtfs/energy_potential_map.html")
print("Map saved to data/oebb_gtfs/energy_potential_map.html")


Map saved to data/oebb_gtfs/energy_potential_map.html


Since there are lines that also travel outside Austria, we should only calculate the part of the journey within Austria.

In [20]:
import geopandas as gpd
from shapely.geometry import LineString, Point

# ============================================================
# Load Austria boundary polygon (robust for GeoPandas ≥1.0)
# ============================================================

url = "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson"

world = gpd.read_file(url)
austria = world[world["ADMIN"] == "Austria"].to_crs(epsg=4326)

print(f"Loaded Austria boundary polygon ({len(austria)} feature).")

# ============================================================
# Create route LineStrings from start/end coords
# ============================================================

# Keep only rows with valid coordinates
geo_df = top_candidates.dropna(subset=['start_lat', 'start_lon', 'end_lat', 'end_lon']).copy()

# Create LineString geometry
geo_df['geometry'] = geo_df.apply(
    lambda r: LineString([(r.start_lon, r.start_lat), (r.end_lon, r.end_lat)]),
    axis=1
)

geo_df = gpd.GeoDataFrame(geo_df, geometry='geometry', crs='EPSG:4326')

# ============================================================
# Step 3 — Intersect with Austria polygon
# ============================================================

# Reproject to a metric CRS for distance calculations (Europe Lambert)
geo_df = geo_df.to_crs(epsg=3035)
austria = austria.to_crs(epsg=3035)

# Compute intersection
geo_df['geometry_AT'] = geo_df['geometry'].intersection(austria.geometry.iloc[0])

# Compute total length and length inside Austria (in km)
geo_df['distance_total_km'] = geo_df['geometry'].length / 1000
geo_df['distance_AT_km'] = geo_df['geometry_AT'].length / 1000

print("Computed route length inside Austria (km).")
geo_df[['start_station', 'end_station', 'distance_total_km', 'distance_AT_km']].head()

# Replace missing Austrian distances (NaN) with 0
geo_df['distance_AT_km'] = geo_df['distance_AT_km'].fillna(0)

# Recalculate energy potential index based on Austrian segment
epi_factor = 1e-6
geo_df['EPI_AT'] = (
    epi_factor *
    geo_df['weekly_total'] *
    geo_df['estimated_weight_t'] *
    (geo_df['avg_speed_kmh'] ** 2) /
    geo_df['distance_AT_km'].replace(0, np.nan)
)

geo_df = geo_df.sort_values('EPI_AT', ascending=False)
geo_df[['start_station', 'end_station', 'distance_AT_km', 'EPI_AT']].head(10)


Loaded Austria boundary polygon (1 feature).
Computed route length inside Austria (km).


/Users/marcelgandler/miniconda3/envs/REE/lib/python3.13/site-packages/shapely/set_operations.py:168: RuntimeWarning: invalid value encountered in intersection
  return lib.intersection(a, b, **kwargs)


,start_station,end_station,distance_AT_km,EPI_AT
3136,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.982843,75.999406
3134,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983091,75.994677
3135,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983112,75.994269
3142,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983215,75.992308
3139,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983321,75.990292
3140,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983463,75.987578
3141,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983484,75.987170
3137,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983568,75.985563
3138,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.983590,75.985155
3145,Kufstein Bahnhof,Nürnberg Hauptbahnhof,3.997832,75.714454


**Exporting the data to CSV**

In [21]:
# ============================================================
# Export results (Austrian distances + EPI) to CSV
# ============================================================

# Select useful columns (exclude geometry objects)
output_columns = [
    'start_station',
    'end_station',
    'estimated_weight_t',
    'avg_speed_kmh',
    'weekly_total',
    'distance_total_km',
    'distance_AT_km',
    'EPI_AT'
]

# Create a plain DataFrame without geometry columns
export_df = geo_df[output_columns].copy()

# Save to CSV
output_path = "data/oebb_gtfs/train_energy_potential_austria.csv"
export_df.to_csv(output_path, index=False)

print(f"CSV saved: {output_path}")
print(export_df.head(10))


CSV saved: data/oebb_gtfs/train_energy_potential_austria.csv
         start_station            end_station  estimated_weight_t  \
3136  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3134  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3135  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3142  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3139  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3140  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3141  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3137  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3138  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   
3145  Kufstein Bahnhof  Nürnberg Hauptbahnhof               400.0   

      avg_speed_kmh  weekly_total  distance_total_km  distance_AT_km  \
3136      66.523277         171.0         221.914377        3.982843   
3134      66.523277         171.0  

We are using the average speed of the whole journey also for the section of the journey in asutria. this is an issue as i.e. the whole journey is 225 km but only 3 km are in austria, therefore the average speed in austria only is not the same as for the 225 kms.

In [22]:
# ============================================================
# Adjust average speed for Austrian segment
# ============================================================

geo_df['avg_speed_AT_kmh'] = (
    geo_df['avg_speed_kmh'] *
    np.sqrt(
        geo_df['distance_AT_km'].clip(lower=0.1) /
        geo_df['distance_total_km'].clip(lower=0.1)
    )
)

# Optional: cap speeds to realistic limits
geo_df['avg_speed_AT_kmh'] = geo_df['avg_speed_AT_kmh'].clip(upper=230)

# Recompute EPI based on adjusted speed
epi_factor = 1e-6
geo_df['EPI_AT'] = (
    epi_factor *
    geo_df['weekly_total'] *
    geo_df['estimated_weight_t'] *
    (geo_df['avg_speed_AT_kmh'] ** 2) /
    geo_df['distance_AT_km'].replace(0, np.nan)
)

geo_df = geo_df.sort_values('EPI_AT', ascending=False)

print("Updated EPI_AT using adjusted average speed for Austrian segment.")
geo_df[['start_station', 'end_station', 'distance_AT_km', 'avg_speed_AT_kmh', 'EPI_AT']].head(10)


Updated EPI_AT using adjusted average speed for Austrian segment.


,start_station,end_station,distance_AT_km,avg_speed_AT_kmh,EPI_AT
5,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.313629,100.843611,18.579550
6,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.315513,100.897066,18.572624
1,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.315841,100.796576,18.531022
2,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.317692,100.850086,18.524648
7,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.321694,101.003040,18.524615
3,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.323856,100.956170,18.477201
4,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.325253,100.987856,18.469319
0,Spielfeld Tarifpunkt Grenze (Bahn),Spielfeld-Straß Bahnhof,1.327505,100.940970,18.420859
79,Salzburg Hauptbahnhof,Freilassing Bahnhof,5.527324,50.967283,17.209809
80,Salzburg Hauptbahnhof,Freilassing Bahnhof,5.527512,50.967283,17.209223


In [23]:
# ============================================================
# Export results (Austrian distances + EPI) to CSV
# ============================================================

# Select useful columns (exclude geometry objects)
output_columns = [
    'start_station',
    'end_station',
    'estimated_weight_t',
    'avg_speed_kmh',
    'weekly_total',
    'distance_total_km',
    'distance_AT_km',
    'avg_speed_AT_kmh',
    'EPI_AT'
]

# Create a plain DataFrame without geometry columns
export_df = geo_df[output_columns].copy()

# Save to CSV
output_path = "data/oebb_gtfs/train_energy_potential_austria.csv"
export_df.to_csv(output_path, index=False)

print(f"CSV saved: {output_path}")
print(export_df.head(10))


CSV saved: data/oebb_gtfs/train_energy_potential_austria.csv
                         start_station              end_station  \
5   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
6   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
1   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
2   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
7   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
3   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
4   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
0   Spielfeld Tarifpunkt Grenze (Bahn)  Spielfeld-Straß Bahnhof   
79               Salzburg Hauptbahnhof      Freilassing Bahnhof   
80               Salzburg Hauptbahnhof      Freilassing Bahnhof   

    estimated_weight_t  avg_speed_kmh  weekly_total  distance_total_km  \
5           400.000000     128.496555           6.0           2.132844   
6           400.000000     128.496555           6.0  

**Calculate Avg Speed Per Stop per Trip**

In [24]:
#merge trips_full with stop_times_geo to get start and endstation
stop_times_geo_with_startEnd = stop_times_geo.merge(
    trips_full[['trip_id', 'start_station', 'end_station']],
    on='trip_id',
    how='left'
)


stop_times_geo_with_startEnd = stop_times_geo_with_startEnd.merge(
    stops[['stop_name' , 'stop_lat', 'stop_lon']],
    on=["stop_lat", "stop_lon"],  # match by latitude & longitude
    how="left"
)

# Parse as elapsed time since service day start
td = pd.to_timedelta(stop_times_geo_with_startEnd["arrival_time"])

# Optional: keep useful pieces
stop_times_geo_with_startEnd["arrival_td"] = td                              # Timedelta
stop_times_geo_with_startEnd["arrival_day_offset"] = (td // pd.Timedelta('1D')).astype(int)
stop_times_geo_with_startEnd["arrival_clock"] = (td % pd.Timedelta('1D'))    # 00:00–23:59:59


grouped = stop_times_geo_with_startEnd.groupby("trip_id")

# Calculate average speed per stop per trip
def calculate_avg_speed(group):
    group = group.sort_values("stop_sequence")
    group["distance"] = group["shape_dist_traveled"].diff()
    group["distance"] = group["distance"] /1000  #convert to km
    #time delta in hours
    group["time"] = group["arrival_td"].dt.total_seconds().diff() / 3600
    #speed in km/h
    group["speed"] = group["distance"] / group["time"]
    return group

stop_times_geo_with_startEnd_calc = stop_times_geo_with_startEnd.groupby("trip_id").apply(calculate_avg_speed)

#for isna in distance time and speed fill with 0
stop_times_geo_with_startEnd_calc['distance'] = stop_times_geo_with_startEnd_calc['distance'].fillna(0)
stop_times_geo_with_startEnd_calc['time'] = stop_times_geo_with_startEnd_calc['time'].fillna(0)
stop_times_geo_with_startEnd_calc['speed'] = stop_times_geo_with_startEnd_calc['speed'].fillna(0)
#drop index 0 column
stop_times_geo_with_startEnd_calc = stop_times_geo_with_startEnd_calc.reset_index(drop=True)

/var/folders/v5/_44s311d71j55ws_3vlqrh040000gn/T/ipykernel_27920/3618681702.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  stop_times_geo_with_startEnd_calc = stop_times_geo_with_startEnd.groupby("trip_id").apply(calculate_avg_speed)


In [25]:
#merge trips full with stop_times_geo_with_startEnd_calc on trip id to get route id and other info
stop_times_geo_with_startEnd_calc[["trip_id"]]
trips_full[["trip_id"]]

stop_times_geo_with_startEnd_calc = stop_times_geo_with_startEnd_calc.merge(
    trips_full[["trip_id","route_id","service_id","trip_short_name","route_short_name","direction_id","block_id","agency_id","route_type","monday","tuesday","wednesday","thursday","friday","saturday","sunday","start_date","end_date"]],
    on='trip_id',
    how='left'
)

In [74]:
def make_Cord_clean(df, stop_col="stop_name", lat_col="stop_lat", lon_col="stop_lon"):
    """
    For each unique stop_name, find the first latitude/longitude values and
    set all occurrences of that stop_name to those coordinates.

    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame containing stop coordinates.
    stop_col : str, default "stop_name"
        Column name for stop identifiers.
    lat_col : str, default "stop_lat"
        Column name for latitude.
    lon_col : str, default "stop_lon"
        Column name for longitude.

    Returns
    -------
    pd.DataFrame
        A copy of the DataFrame with cleaned coordinates.
    """
    df = df.copy()

    # Create lookup table of unique stop_name → first lat/lon
    first_coords = (
        df.dropna(subset=[stop_col, lat_col, lon_col])
          .drop_duplicates(subset=[stop_col])
          .set_index(stop_col)[[lat_col, lon_col]]
    )

    # Map these consistent coordinates back into the full DataFrame
    df[lat_col] = df[stop_col].map(first_coords[lat_col])
    df[lon_col] = df[stop_col].map(first_coords[lon_col])

    return df

stop_times_geo_with_startEnd_calc = make_Cord_clean(stop_times_geo_with_startEnd_calc)

In [88]:
# --- Load world countries polygons (GeoPandas 1.0+ compatible) ---
import geopandas as gpd

def load_world_countries():
    """
    Returns a GeoDataFrame with columns: ['name', 'iso_a3', 'geometry'].
    Tries geodatasets first, then cartopy as a fallback.
    """
    # Try geodatasets (pip install geodatasets)
    try:
        from geodatasets import get_path
        # Admin-0 countries layer
        world_path = get_path("naturalearth.cultural.admin_0_countries")
        world = gpd.read_file(world_path)[["NAME_EN", "ADM0_A3", "geometry"]]
        world = world.rename(columns={"NAME_EN": "name", "ADM0_A3": "iso_a3"})
        return world
    except Exception:
        pass

    # Fallback: cartopy (pip install cartopy)
    try:
        import cartopy.io.shapereader as shpreader
        shp = shpreader.natural_earth(
            resolution="110m",
            category="cultural",
            name="admin_0_countries"
        )
        world = gpd.read_file(shp)[["NAME_EN", "ADM0_A3", "geometry"]]
        world = world.rename(columns={"NAME_EN": "name", "ADM0_A3": "iso_a3"})
        return world
    except Exception as e:
        raise RuntimeError(
            "Could not load Natural Earth admin_0_countries. "
            "Install one of:\n"
            "  pip install geodatasets\n"
            "  pip install cartopy\n"
            f"Underlying error: {e}"
        )

world = load_world_countries()
import geopandas as gpd
from shapely.geometry import Point


gdf = gpd.GeoDataFrame(
    stop_times_geo_with_startEnd_calc,
    geometry=gpd.points_from_xy(stop_times_geo_with_startEnd_calc["stop_lon"], stop_times_geo_with_startEnd_calc["stop_lat"]),
    crs="EPSG:4326"
)

# Spatial join: assign country to each point
joined = gpd.sjoin(gdf, world, how="left", predicate="within")

stop_times_geo_with_startEnd_calc["country_stop"] = joined["name"]
stop_times_geo_with_startEnd_calc["country_iso3_stop"] = joined["iso_a3"]

stop_times_geo_with_startEnd_calc.drop(columns=["country", "country_iso3"], inplace=True)
stop_times_geo_with_startEnd_calc

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,stop_lat,...,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date,country_stop,country_iso3_stop
0,1.TA.1-MB4-j25-1.1.R,21:04:00,21:04:00,at:48:134:0:8,1,NaN,0,0,0.00,47.079585,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
1,1.TA.1-MB4-j25-1.1.R,21:05:00,21:05:00,at:48:892:0:1,2,NaN,0,0,997.90,47.079120,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
2,1.TA.1-MB4-j25-1.1.R,21:07:00,21:07:00,at:48:139:0:3,3,NaN,0,0,2443.60,47.085384,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
3,1.TA.1-MB4-j25-1.1.R,21:10:00,21:10:00,at:48:326:0:3,4,NaN,0,0,4565.81,47.098771,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
4,1.TA.1-MB4-j25-1.1.R,21:13:00,21:14:00,at:48:187:0:3,5,NaN,0,0,5832.83,47.109967,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109837,99.TA.4-CX9-W-j25-1.20.H,24:24:00,24:24:00,at:43:8767:0:2,39,NaN,0,0,107205.35,47.659914,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
109838,99.TA.4-CX9-W-j25-1.20.H,24:31:00,24:32:00,at:43:8769:0:3,41,NaN,0,0,112993.58,47.639569,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
109839,99.TA.4-CX9-W-j25-1.20.H,24:37:00,24:37:00,at:46:7428:0:4,42,NaN,0,0,117337.65,47.620850,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT
109840,99.TA.4-CX9-W-j25-1.20.H,24:41:00,24:42:00,at:46:7427:0:2,43,NaN,0,0,120195.91,47.616145,...,1,1,1,1,1,1,20241215,20251213,Austria,AUT


**Mapping of Routed Trips**

In [ ]:

smaller_df = stop_times_geo_with_startEnd_calc[:98]
#make black and white folium map
m = folium.Map(location=[47.5, 14.5], zoom_start=7, tiles = "CartoDB positron")  # Center on Austria

# Austria bounding box
DRAW_MARKERS = True

#make all color markers small red dots
stops_color = "#FF0000"
route_color= "#000000"

routes = smaller_df["route_id"].astype(str).unique()
route_names = smaller_df["route_short_name"].astype(str).unique()

linestyles = {}
line_styles_list = ["solid", "dashed", "dotted", "dashdot"]

def get_linestyle(route_name, default_color=route_color):
    """
    Returns (linestyle, color) for a given route name string.
    """
    rn = str(route_name).upper()

    # --- Commuter / S-Bahn type ---
    if rn.startswith("S") or rn.startswith("CAT"):   # CAT = City Airport Train
        return "dashdot", "#2A98E2"

    # --- Regional services (REX, RX, R, SV etc.) ---
    if any(rn.startswith(prefix) for prefix in ["REX", "RX", "R", "SV"]):
        return "dashed", default_color

    # --- Intercity / Long-distance (RJ, RJX, CJX, IC, EC, EN, WB, A*, D*, CH*, CZ*, HU*, SI*) ---
    if any(rn.startswith(prefix) for prefix in [
        "RJ", "RJX", "CJX", "IC", "EC", "EN", "WB", "A", "D", "CH", "CZ", "HU", "SI"
    ]):
        return "solid", default_color

    # --- Default fallback ---
    return "solid", "#808080"




for r in routes:
    r_group = folium.FeatureGroup(name=f"Route {r}", show=True)
    m.add_child(r_group)

    g_route = smaller_df[smaller_df["route_id"].astype(str) == r].copy()
    g_route.sort_values(["trip_id", "stop_sequence"], inplace=True)

    
    for _, row in g_route.iterrows():
        print(row['route_id'])
        lat = row['stop_lat']
        lon = row['stop_lon']
        linestyles[row['route_short_name']] = {'linestyle': get_linestyle(row['route_short_name'])[0]}
        if DRAW_MARKERS:
            #red markers with black border
            folium.CircleMarker(
                location=(lat, lon),
                radius=2.6,
                color=stops_color,
                fill=True,
                fill_color=stops_color,
                fill_opacity=1.0,
                stroke=True,
                weight=0,
                popup=folium.Popup(row['stop_name'])
                ).add_to(m)

    # Draw all segments per trip, but only if both endpoints lie inside Austria bbox
    for trip_id, g_trip in g_route.groupby("trip_id"):
        # Draw the route line
        points = list(zip(g_trip["stop_lat"], g_trip["stop_lon"]))
        folium.PolyLine(
            locations=points,
            color=route_color,
            weight=2,
            opacity=0.7,
            popup=f"Trip ID: {trip_id}"
        ).add_to(m)

# Save map to HTML
m.save("data/oebb_gtfs/oebb_routes_map.html")
print("Map saved to data/oebb_gtfs/oebb_routes_map.html")

1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MB4-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-MS1-V-j25-1
1-S1-K-j25-1
1-S1-K-j25-1
1-S1-K-j25-1
1-S1-M-j25-1
1-S1-M-j25-1
1-S1-M-j25-1
1-S1-M-j25-1
1-S1-M-j25-1
1-S1-M-j25-1
1-S1-O-j25-1
1-S1-O-j25-1
1-S1-O-j25-1
1-S1-O-j25-1
1-S1-O-j25-1
1-S1-O-j25-1
1-S1-O-j25-1
1-S1-V-j25-1
1-S1-V-j25-1
1-S1-V-j25-1
1-S1-V-j25-1
1-S1-V-j25-1
1-S1-W-j25-1
1-S1-W-j25-1
1-S1-W-j25-1
1-S1-W-j25-1
1-S1-W-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-K-j25-1
1-S2-O-j25-1
1-S2-O-j25-1
1-S2-O-j25-1
1-S2-O-j25-1
1-S2-O-j25-1
1-S2-O-j25-1
1-S2-O-j25-1
1-S2-S-j25-

In [33]:

smaller_df = stop_times_geo_with_startEnd_calc[:200]
m = folium.Map(location=[47.5, 14.5], zoom_start=7)

# Austria bounding box
AT_LAT_MIN, AT_LAT_MAX = 46.372, 49.021
AT_LON_MIN, AT_LON_MAX =  9.530, 17.162
DRAW_MARKERS = True

# Generate distinct colors for each route
def route_color_palette(n):
    random.seed(42)
    colors = []
    for i in range(n):
        hue = (i * 0.61803398875) % 1.0
        import colorsys
        r, g, b = colorsys.hsv_to_rgb(hue, 0.6, 0.95)
        colors.append('#%02x%02x%02x' % (int(r*255), int(g*255), int(b*255)))
    return colors
# Routes & colors based on route ID
routes = smaller_df["route_id"].astype(str).unique().tolist()
route_segment= smaller_df["route_short_name"].astype(str).tolist()
route_colors = route_color_palette(len(route_segment))
route_to_color = dict(zip(routes, route_colors))

# Keep track of coordinates that pass the AT filter to recenter if needed
kept_lats, kept_lons = [], []

for r in routes:
    r_group = folium.FeatureGroup(name=f"Route {r}", show=True)
    m.add_child(r_group)

    g_route = smaller_df[smaller_df["route_id"].astype(str) == r].copy()
    g_route.sort_values(["trip_id", "stop_sequence"], inplace=True)

    # Optional stop markers (colored by route)
    if DRAW_MARKERS:
        for _, row in g_route.iterrows():
            lat, lon = float(row["stop_lat"]), float(row["stop_lon"])
            if (AT_LAT_MIN <= lat <= AT_LAT_MAX) and (AT_LON_MIN <= lon <= AT_LON_MAX):
                kept_lats.append(lat); kept_lons.append(lon)
                folium.CircleMarker(
                    location=[lat, lon],
                    radius=3,
                    color=route_to_color[r],
                    weight=1,
                    fill=True,
                    fill_opacity=0.9,
                    opacity=0.9,
                    tooltip=f"Route {r} • Trip {row['trip_id']} • Seq {row['stop_sequence']}"
                ).add_to(r_group)

    # Draw all segments per trip, but only if both endpoints lie inside Austria bbox
    for trip_id, g_trip in g_route.groupby("trip_id"):
        g_trip = g_trip.sort_values("stop_sequence")
        if len(g_trip) < 2:
            continue

        # Base segments (route color)
        for i in range(len(g_trip) - 1):
            lat1, lon1 = float(g_trip.iloc[i]["stop_lat"]),   float(g_trip.iloc[i]["stop_lon"])
            lat2, lon2 = float(g_trip.iloc[i+1]["stop_lat"]), float(g_trip.iloc[i+1]["stop_lon"])

            inside1 = (AT_LAT_MIN <= lat1 <= AT_LAT_MAX) and (AT_LON_MIN <= lon1 <= AT_LON_MAX)
            inside2 = (AT_LAT_MIN <= lat2 <= AT_LAT_MAX) and (AT_LON_MIN <= lon2 <= AT_LON_MAX)
            if not (inside1 and inside2):
                continue  # skip any segment that exits Austria

            kept_lats += [lat1, lat2]; kept_lons += [lon1, lon2]

            sp = g_trip.iloc[i]["speed"]
            tip = f"Route {r} • Trip {trip_id} • Seq {g_trip.iloc[i]['stop_sequence']}→{g_trip.iloc[i+1]['stop_sequence']} • {sp}"
            folium.PolyLine(
                locations=[[lat1, lon1], [lat2, lon2]],
                color=route_to_color[r],
                weight=3,
                opacity=0.7,
                tooltip=tip
            ).add_to(r_group)

        # Highlight highest-speed segment (BRIGHT green), still respecting Austria filter
        seg_vals = []
        for i in range(len(g_trip)-1):
            v = g_trip.iloc[i]["speed"]
            seg_vals.append(v if np.isfinite(v) else -np.inf)

        if any(np.isfinite(seg_vals)):
            max_idx = int(np.nanargmax(seg_vals))
            lat1, lon1 = float(g_trip.iloc[max_idx]["stop_lat"]),   float(g_trip.iloc[max_idx]["stop_lon"])
            lat2, lon2 = float(g_trip.iloc[max_idx+1]["stop_lat"]), float(g_trip.iloc[max_idx+1]["stop_lon"])

            inside1 = (AT_LAT_MIN <= lat1 <= AT_LAT_MAX) and (AT_LON_MIN <= lon1 <= AT_LON_MAX)
            inside2 = (AT_LAT_MIN <= lat2 <= AT_LAT_MAX) and (AT_LON_MIN <= lon2 <= AT_LON_MAX)
            # Oops! fix a typo: use AT_LAT_MAX for lat2 check
            inside2 = (AT_LAT_MIN <= lat2 <= AT_LAT_MAX) and (AT_LON_MIN <= lon2 <= AT_LON_MAX)

            if inside1 and inside2:
                vmax = seg_vals[max_idx]
                tip = (f"HIGHEST SPEED • Route {r} • Trip {trip_id} • "
                       f"Seq {g_trip.iloc[max_idx]['stop_sequence']}→{g_trip.iloc[max_idx+1]['stop_sequence']} • {vmax}")
                folium.PolyLine(
                    locations=[[lat1, lon1], [lat2, lon2]],
                    color="#00FF00",   # BRIGHT green
                    weight=6,
                    opacity=0.95,
                    tooltip=tip
                ).add_to(r_group)

# Recenter if we filtered out lots of points
if kept_lats and kept_lons:
    m.location = [float(np.mean(kept_lats)), float(np.mean(kept_lons))]
    m.zoom_start = 8

# Legend
legend_html = """
<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: white; padding: 10px; border: 1px solid #ccc; border-radius: 8px; max-height: 40vh; overflow: auto;">
  <h4 style="margin:0 0 6px 0;">Routes (Greyscale Map)</h4>
  <div style="font-size: 12px;">
"""
for r in routes:
    legend_html += f'<div style="display:flex;align-items:center;margin-bottom:4px;"><span style="display:inline-block;width:14px;height:14px;background:{route_to_color[r]};margin-right:6px;border:1px solid #666;"></span>{r}</div>'
legend_html += """
    <hr style="margin:6px 0;" />
    <div><span style="display:inline-block;width:14px;height:14px;background:#00FF00;margin-right:6px;border:1px solid #666;"></span>Highest-speed segment</div>
    <div style="margin-top:6px; font-style: italic;">Only segments fully inside Austria are shown.</div>
  </div>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)
folium.TileLayer('CartoDB positron', name='Greyscale Map', control=False).add_to(m)
#make other layers semi-transparent
for key, layer in m._children.items():
    if isinstance(layer, folium.raster_layers.TileLayer) and layer.tile_name != 'Greyscale Map':
        layer.options['opacity'] = 0.4
# Save / display:
m.save("./data/oebb_gtfs/route_stops_speed_map.html")
# m  # in notebooks

Adding the number of trips per origin-destination

In [3]:
import pandas as pd

df = pd.read_csv("data/oebb_gtfs/stop_times_geo_with_startEnd_calc.csv")

weekly_counts = (
    df.groupby(['start_station', 'end_station'])['trip_id']
    .nunique()
    .reset_index(name='weekly_total')
)

df = df.merge(weekly_counts, on=['start_station', 'end_station'], how='left')

df[['trip_id', 'start_station', 'end_station', 'weekly_total']].head(10)

df.to_csv("data/oebb_gtfs/stop_times_geo_with_startEnd_calc1.csv", index=False)


In [4]:
df[['trip_id', 'start_station', 'end_station', 'weekly_total']].head(10)


,trip_id,start_station,end_station,weekly_total
0,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
1,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
2,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
3,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
4,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
5,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
6,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
7,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
8,1.TA.1-MB4-j25-1.1.R,Schruns Bahnhof,Bludenz Bahnhof,41
9,1.TA.1-MS1-V-j25-1.1.R,Bludenz Bahnhof,Bregenz Bahnhof,5


In [9]:
df = pd.read_csv("data/oebb_gtfs/stop_times_geo_with_startEnd_calc1.csv")

epi_factor = 1e-6  # scaling constant

df['EPI_segment'] = (
    epi_factor *
    df['weekly_total'] *
    df['estimated_weight_t'] *
    (df['speed'] ** 2) /
    df['distance'].replace(0, pd.NA)
).fillna(0)


df.to_csv("data/oebb_gtfs/stop_times_geo_with_startEnd_calc1.csv", index=False)


C:\Users\fatos\AppData\Local\Temp\ipykernel_16444\3762313406.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ).fillna(0)
